In [0]:
# import statements
from pyspark.sql.functions import *

In [0]:
# Config & Parameters


# Path where I uploaded the 3 CSV files

RAW_FILES_PATH = "/Volumes/invoices/raw/batch_1/"

# Delta table target
CATALOG        = "invoices"
BRONZE_SCHEMA  = "bronze"
BRONZE_TABLE   = "raw_invoices"
FULL_TABLE     = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"

# Checkpoint path for Auto Loader (tracks which files were already loaded)


# Batch identifier — change this per run or drive it from a Workflow parameter
BATCH_ID = "batch_1"

print(f"Source  : {RAW_FILES_PATH}")
print(f"Target  : {FULL_TABLE}")
print(f"Batch   : {BATCH_ID}")

In [0]:

# ============================================================
# Create Catalog & Schema (idempotent)
# ============================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

print("Catalog and schema ready.")

In [0]:
# ============================================================
# STEP 2 — Read just the first file to inspect columns
# ============================================================

df_test = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")   # keep everything as string for now
    .option("sep", ",")               # explicitly set delimiter
    .option("quote", '"')             # explicitly set quote char
    .option("escape", '"')
    .option("multiLine", "true")      # set true in case values span multiple lines
    .load("/Volumes/invoices/raw/batch_1/batch1_1.csv")  # load ONE file first
)

# Print actual column names Spark detected
print("Detected columns:")
for col_name in df_test.columns:
    print(f"  → '{col_name}'")

display(df_test.limit(3))

In [0]:
# STEP 1 — See the raw file content exactly as it is
raw_text = spark.read.text("/Volumes/invoices/raw/batch_1/batch1_1.csv")
display(raw_text.limit(5))

In [0]:
# ============================================================
# Batch Ingestion
# Reads all CSVs from the Volume folder in one shot
# ============================================================

from pyspark.sql.functions import current_timestamp, lit, col

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # keep as string first, Json shouldnt be inferred
    .option("sep", ",")               # explicitly set delimiter
    .option("quote", '"')             # explicitly set quote char
    
    .option("escape", '"')
    .option("multiLine", "true").   # multiline json
    option("encoding", "UTF-8")
    .load(RAW_FILES_PATH)          # Volume path works directly here
)

# Add Bronze audit metadata columns
df_bronze = (
    df_raw
    .withColumn("source_file",     col("_metadata.file_path"))
    .withColumn("ingested_at",     current_timestamp())
    .withColumn("batch_id",        lit(BATCH_ID))
    .withColumn("pipeline_layer",  lit("bronze"))
)

print(f"Rows read: {df_bronze.count()}")
df_bronze.printSchema()
display(df_bronze.limit(5))

In [0]:
# STEP 3 — Fix column names + add metadata + write to Bronze

from pyspark.sql.functions import current_timestamp, lit, input_file_name

df_bronze = (
    df_raw
    # Fix column names — replace spaces with underscores
    .withColumnRenamed("File Name",   "file_name")
    .withColumnRenamed("Json Data",   "json_data")
    .withColumnRenamed("OCRed Text",  "ocred_text")
    # Add audit metadata columns
    .withColumn("source_file",    col("_metadata.file_path"))
    .withColumn("source_file_name", col("_metadata.file_name")) 
    .withColumn("ingested_at",    current_timestamp())
    .withColumn("ingestion_date",   to_date(current_timestamp()))  # partition column — DATE only
    .withColumn("batch_id",       lit(BATCH_ID))
    .withColumn("pipeline_layer", lit("bronze"))
)

# Verify columns before writing
print("Final columns:", df_bronze.columns)
display(df_bronze.limit(3))

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

df_bronze = (
    df_raw
    .withColumnRenamed("File Name",  "file_name")
    .withColumnRenamed("Json Data",  "json_data")
    .withColumnRenamed("OCRed Text", "ocred_text")
    .withColumn("source_file",      col("_metadata.file_path"))
    .withColumn("source_file_name", col("_metadata.file_name"))
    .withColumn("ingested_at",      current_timestamp())
    .withColumn("ingestion_date",   to_date(current_timestamp()))
    .withColumn("batch_id",         lit(BATCH_ID))
    .withColumn("pipeline_layer",   lit("bronze"))
)

(
    df_bronze
    .write
    .format("delta")
    .mode("append")                  # back to append — clean table, safe to use
    .option("mergeSchema", "true")
    .partitionBy("ingestion_date")
    .saveAsTable("invoices.bronze.raw_invoices")
)

print(f"Done! Total rows: {spark.table('invoices.bronze.raw_invoices').count()}")

# Verify partition exists
spark.sql("DESCRIBE DETAIL invoices.bronze.raw_invoices").select("partitionColumns").show(truncate=False)

In [0]:
# Print one full JSON value to see ALL fields
sample_json = (
    spark.table("invoices.bronze.raw_invoices")
    .filter(col("json_data").isNotNull())
    .select("json_data")
    .first()[0]
)

print(sample_json)